# Bootcamp Playbook

Welcome to the bootcamp! This three-part notebook series introduces NVIDIA Nemotron reasoning controls, shows how to build a movie database MCP server, and connects that server to the NVIDIA NeMo Agent Toolkit.

## Table of Contents

1. [NVIDIA Nemotron Reasoning Controls](01_reasoning_controls_notebook.ipynb)
2. [Movie Database MCP Server](02_movie_database_mcp.ipynb)
3. [NeMo Agent Toolkit with the Movie MCP Server](03_nemo_agent_toolkit.ipynb)


# NVIDIA Nemotron Reasoning Controls

This self-contained notebook focuses on three controls from the Nemotron 3 Super getting-started example:

1. `enable_thinking: true`
2. `reasoning_budget`
3. `low_effort: true`

The goal is to give workshop participants a compact, hands-on way to compare deeper reasoning, bounded reasoning, and faster low-effort reasoning on the same prompts.

By the end of this notebook, you will be able to:

- configure an OpenAI-compatible client for a Nemotron model hosted by NVIDIA NIM;
- stream reasoning content separately from the final answer;
- compare thinking off, thinking on, bounded reasoning, and low-effort reasoning; and
- select a reasoning mode based on task complexity and latency requirements.

## 1. Configure the API Key, Endpoint, and Model

The notebook defaults to Nemotron 3 Super because that is the model used in the referenced getting-started guide. You can override the model by setting `NEMOTRON_MODEL` before running the notebook.

| Setting | Purpose |
| --- | --- |
| `NVIDIA_API_KEY` | Authenticates requests to the hosted NVIDIA API endpoint. The cell prompts securely if the variable is not already set. |
| `NVIDIA_BASE_URL` | Points the OpenAI client to NVIDIA's OpenAI-compatible chat-completions API. |
| `NEMOTRON_MODEL` | Optionally replaces the default model without editing the notebook. |

The configured client is reused by every experiment so that only the reasoning controls and prompts change.

In [ ]:
import os
from getpass import getpass

from openai import OpenAI


if not os.environ.get("NVIDIA_API_KEY"):
    os.environ["NVIDIA_API_KEY"] = getpass("Enter your NVIDIA API Key: ").strip()

NVIDIA_BASE_URL = "https://integrate.api.nvidia.com/v1/"
MODEL = os.environ.get("NEMOTRON_MODEL", "nvidia/nemotron-3-super-120b-a12b")

client = OpenAI(
    base_url=NVIDIA_BASE_URL,
    api_key=os.environ["NVIDIA_API_KEY"],
    default_headers={"NVCF-POLL-SECONDS": "1800"},
)

print(f"Configured endpoint: {NVIDIA_BASE_URL}")
print(f"Configured model: {MODEL}")

## 2. Create Reusable Streaming Helpers

Reasoning-capable Nemotron responses can stream reasoning content separately from the final answer. The helper below checks both `reasoning_content` and `reasoning`, then prints the reasoning in gray and the final answer in the default color.

| Helper | Purpose |
| --- | --- |
| `make_reasoning_extra_body` | Builds the NVIDIA-specific reasoning-control payload. |
| `preview_payload` | Displays the controls that will be sent with a request. |
| `stream_with_reasoning` | Separates streamed reasoning from answer text and records basic measurements. |
| `run_demo` | Sends a prompt, renders the stream, and returns both text and timing data for comparison. |

The visible reasoning stream is useful for this controlled workshop comparison. Treat it as model-generated content—not as a verified explanation of model internals—and avoid exposing it in a production user interface.

In [ ]:
import json
import time
from typing import Any


GRAY = "\033[90m"
RESET = "\033[0m"


def make_reasoning_extra_body(
    *,
    enable_thinking: bool = True,
    reasoning_budget: int | None = None,
    low_effort: bool | None = None,
) -> dict[str, Any]:
    """Build the extra_body payload for Nemotron reasoning controls."""
    chat_template_kwargs = {"enable_thinking": enable_thinking}

    if low_effort is not None:
        chat_template_kwargs["low_effort"] = low_effort

    extra_body: dict[str, Any] = {"chat_template_kwargs": chat_template_kwargs}

    if reasoning_budget is not None:
        extra_body["reasoning_budget"] = reasoning_budget

    return extra_body


def preview_payload(extra_body: dict[str, Any]) -> None:
    """Print only the reasoning-control portion of the request."""
    print(json.dumps(extra_body, indent=2))


def stream_with_reasoning(completion, *, show_reasoning: bool = True) -> dict[str, Any]:
    """Stream a response, separate reasoning from the final answer, and return both."""
    reasoning = ""
    answer = ""
    started_at = time.perf_counter()
    in_reasoning = False

    for chunk in completion:
        if not getattr(chunk, "choices", None):
            continue

        delta = chunk.choices[0].delta
        reasoning_piece = (
            getattr(delta, "reasoning_content", None)
            or getattr(delta, "reasoning", None)
        )
        content_piece = getattr(delta, "content", None)

        if reasoning_piece:
            reasoning += reasoning_piece
            if show_reasoning:
                if not in_reasoning:
                    print(GRAY, end="")
                    in_reasoning = True
                print(reasoning_piece, end="", flush=True)

        if content_piece:
            answer += content_piece
            if in_reasoning:
                print(RESET, end="")
                in_reasoning = False
            print(content_piece, end="", flush=True)

    if in_reasoning:
        print(RESET, end="")

    print()
    elapsed_seconds = time.perf_counter() - started_at
    return {
        "reasoning": reasoning,
        "answer": answer,
        "reasoning_chars": len(reasoning),
        "answer_chars": len(answer),
        "elapsed_seconds": elapsed_seconds,
    }


def run_demo(
    prompt: str,
    *,
    extra_body: dict[str, Any],
    system_prompt: str = "You are a helpful NVIDIA Nemotron assistant.",
    max_tokens: int = 4096,
    temperature: float = 1.0,
    top_p: float = 0.95,
    show_reasoning: bool = True,
) -> dict[str, Any]:
    """Create a streamed chat completion and display the reasoning and answer."""
    print("Request reasoning controls:")
    preview_payload(extra_body)
    print("\nStreamed response:\n")

    completion = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt},
        ],
        extra_body=extra_body,
        temperature=temperature,
        top_p=top_p,
        max_tokens=max_tokens,
        stream=True,
        timeout=1800,
    )

    result = stream_with_reasoning(completion, show_reasoning=show_reasoning)
    print(
        "\nSummary: "
        f"reasoning_chars={result['reasoning_chars']:,}, "
        f"answer_chars={result['answer_chars']:,}, "
        f"elapsed_seconds={result['elapsed_seconds']:.1f}"
    )
    return result

## 3. Baseline: Thinking Off

This is the fast, direct control run. Run it first so participants can compare how the same prompt behaves when reasoning is enabled.

As you run the cell, observe:

- whether the model answers the timing question correctly;
- how concise the final answer is; and
- the elapsed time reported in the summary.

This result is the baseline for the next experiment.

In [ ]:
comparison_prompt = """
A workshop has three demo stations: inference, reasoning, and agents.
Each station takes 12 minutes. Participants need 3 minutes to move between stations.
Can a group complete all three stations in 45 minutes? Explain briefly.
"""

baseline_result = run_demo(
    comparison_prompt,
    extra_body=make_reasoning_extra_body(enable_thinking=False),
    max_tokens=1024,
    temperature=0,
    top_p=1,
    show_reasoning=False,
)

## 4. `enable_thinking: true`

Turning thinking on asks the model to reason before producing the final user-facing response. In the stream, the reasoning appears separately from the answer.

This cell uses the same prompt as the baseline so that the reasoning mode is the main experimental difference. Compare the answer, reasoning length, and elapsed time with the previous run. A longer reasoning stream does not automatically mean a better final answer; judge the answer against the task itself.

In [ ]:
thinking_result = run_demo(
    comparison_prompt,
    extra_body=make_reasoning_extra_body(enable_thinking=True),
    max_tokens=2048,
    temperature=1.0,
    top_p=0.95,
    show_reasoning=True,
)

## 5. `reasoning_budget`

`reasoning_budget` limits how much reasoning the model can use. Lower budgets are useful for latency-sensitive tasks, while larger budgets are useful for tasks that require more multi-step planning.

The next two cells use the same constrained scheduling prompt with different budgets. Keeping the prompt fixed makes it easier to see whether additional reasoning improves constraint handling enough to justify the extra work.

### Smaller Reasoning Budget

Start with a budget of `1024`. Check whether the response includes every required activity, realistic transitions, five minutes for Q&A, and a total duration of 45 minutes.

In [ ]:
budget_prompt = """
Create a 45-minute hands-on mini-agenda for ML engineers learning Nemotron.
Constraints:
- Include one API warmup, one reasoning-control demo, and one agentic workflow demo.
- Leave 5 minutes for Q&A.
- Keep transitions realistic.
- Return a minute-by-minute agenda and explain the tradeoffs.
"""

small_budget_result = run_demo(
    budget_prompt,
    extra_body=make_reasoning_extra_body(
        enable_thinking=True,
        reasoning_budget=1024,
    ),
    max_tokens=3072,
    show_reasoning=True,
)

### Larger Reasoning Budget

Increase the budget to `8192` while keeping the prompt unchanged. Compare whether the larger budget produces a more complete or internally consistent agenda, then weigh that improvement against reasoning length and elapsed time.

In [ ]:
larger_budget_result = run_demo(
    budget_prompt,
    extra_body=make_reasoning_extra_body(
        enable_thinking=True,
        reasoning_budget=8192,
    ),
    max_tokens=8192,
    show_reasoning=True,
)

## 6. `low_effort: true`

`low_effort` keeps thinking enabled but asks for a shorter, faster reasoning path. It is a good fit when you want a reasoning-capable mode without deep exploration.

Unlike `reasoning_budget`, which sets an explicit bound, `low_effort` requests a lighter reasoning approach. The prompt asks for practical mode-selection advice, so evaluate whether the answer still distinguishes all four modes clearly while using less reasoning time.

In [ ]:
low_effort_prompt = """
A participant asks: should I use reasoning mode for every chatbot request?
Give a practical answer with examples of when to use thinking, bounded thinking,
low-effort thinking, and thinking off.
"""

low_effort_result = run_demo(
    low_effort_prompt,
    extra_body=make_reasoning_extra_body(
        enable_thinking=True,
        low_effort=True,
    ),
    max_tokens=2048,
    show_reasoning=True,
)

## 7. Compare the Runs

The exact numbers vary by run, but this table makes the trade-offs visible: reasoning length, answer length, and elapsed time.

Use these measurements as comparison signals rather than quality scores:

- character counts are easy-to-read proxies for response length, not token counts or cost;
- elapsed time can vary with network and service load; and
- answer quality must still be judged against the prompt's requirements.

A useful production choice balances correctness and completeness with acceptable latency—not simply the shortest or longest reasoning stream.

In [ ]:
results = [
    ("thinking_off", baseline_result),
    ("thinking_on", thinking_result),
    ("budget_1024", small_budget_result),
    ("budget_8192", larger_budget_result),
    ("low_effort", low_effort_result),
]

header = f"{'run':<20} {'reasoning_chars':>16} {'answer_chars':>14} {'elapsed_seconds':>16}"
print(header)
print("-" * len(header))

for name, result in results:
    print(
        f"{name:<20} "
        f"{result['reasoning_chars']:>16,} "
        f"{result['answer_chars']:>14,} "
        f"{result['elapsed_seconds']:>16.1f}"
    )

## 8. Workshop Recipes

Use these request shapes as a quick reference during the demo.

| Recipe | When to try it |
| --- | --- |
| `direct_answer` | Simple extraction, classification, or short factual responses. |
| `thinking_on` | Complex tasks where exploration is more important than predictable latency. |
| `bounded_thinking` | Multi-step tasks that need reasoning within a defined budget. |
| `low_effort_thinking` | Moderate tasks where a shorter reasoning path is preferred. |
| `low_effort_with_budget` | Latency-sensitive reasoning with both a lighter mode and an explicit bound. |

These are starting points for experimentation rather than universal defaults.

In [ ]:
recipes = {
    "direct_answer": make_reasoning_extra_body(enable_thinking=False),
    "thinking_on": make_reasoning_extra_body(enable_thinking=True),
    "bounded_thinking": make_reasoning_extra_body(
        enable_thinking=True,
        reasoning_budget=4096,
    ),
    "low_effort_thinking": make_reasoning_extra_body(
        enable_thinking=True,
        low_effort=True,
    ),
    "low_effort_with_budget": make_reasoning_extra_body(
        enable_thinking=True,
        reasoning_budget=2048,
        low_effort=True,
    ),
}

for name, payload in recipes.items():
    print(f"\n{name}")
    print(json.dumps(payload, indent=2))

## 9. Participant Exercise

Change the prompt and recipe below. Before running it, predict which mode will offer the best trade-off between latency and quality.

1. Choose a prompt with at least two constraints that can be checked objectively.
2. Select a recipe and write down why you expect it to fit the task.
3. Run the cell and compare the result with your prediction.
4. Try one alternative recipe if the answer misses a constraint or uses more reasoning than the task appears to need.

In [ ]:
my_prompt = """
Design a two-slide explanation of reasoning_budget for an engineering audience.
Slide 1 should explain the control. Slide 2 should explain when to tune it.
"""

my_recipe = recipes["low_effort_with_budget"]

my_result = run_demo(
    my_prompt,
    extra_body=my_recipe,
    max_tokens=3072,
    show_reasoning=True,
)

## Takeaways

- Use `enable_thinking: true` for complex reasoning, planning, logic, and technical problem-solving.
- Use `reasoning_budget` to bound reasoning work and keep latency and cost predictable.
- Use `low_effort: true` when you want reasoning mode with shorter, cheaper responses.
- Use `enable_thinking: false` for simple responses where direct output matters more than reasoning depth.

For most production-facing apps, keep raw reasoning out of the user interface and render only the final answer unless the demo or debugging workflow explicitly needs to inspect the reasoning stream.

## Optional Challenge: Design a Reasoning Policy

The notebook selects reasoning controls manually. Create a small routing function that chooses a recipe based on the task, then evaluate whether the choice gives an appropriate latency-quality trade-off.

| Task | What it exercises |
| --- | --- |
| Define a familiar technical term in one sentence. | Thinking off for a direct response |
| Recommend one of two tools when latency matters. | Low-effort thinking |
| Build a constrained 30-minute workshop agenda. | A bounded reasoning budget |
| Compare two agent architectures and defend a choice. | Deeper reasoning |

Run one prompt from each row, record the selected recipe and elapsed time, and score the answers for correctness, completeness, and clarity. Finish by explaining which routing rule you would use in a production application and why.

## Links and Resources

- **[NVIDIA Nemotron 3 Super Model Card](https://build.nvidia.com/nvidia/nemotron-3-super-120b-a12b)**: Model details and a hosted API example for `nvidia/nemotron-3-super-120b-a12b`.
- **[NVIDIA NIM Thinking Budget Control](https://docs.nvidia.com/nim/large-language-models/1.15.0/thinking-budget-control.html)**: Guidance for configuring `reasoning_budget`, `low_effort`, and model-specific reasoning controls.
- **[NVIDIA RAG Blueprint Reasoning Controls](https://docs.nvidia.com/rag/latest/enable-nemotron-thinking.html)**: An overview of Nemotron reasoning modes and their latency and quality trade-offs.
- **[OpenAI Python Streaming Helpers](https://github.com/openai/openai-python/blob/main/helpers.md)**: Streaming patterns for the OpenAI-compatible Python client used in this notebook.

---

## Licensing

Copyright © 2026 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.